##Importing functions

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

##Importing data from Silver Tables

In [0]:
df_marvel_movies = spark.read.format('delta').load('/FileStore/tables/Marvel/Silver/Marvel_Movies/')
df_performance = spark.read.format('delta').load('/FileStore/tables/Marvel/Silver/BoxOffice_Performance/')
df_public_response = spark.read.format('delta').load('/FileStore/tables/Marvel/Silver/Public_Response/')

In [0]:
df_marvel_movies = df_marvel_movies.select('id','name')

##Joins between Silver Dataframes

In [0]:
df_silver = df_marvel_movies.join(df_performance,df_marvel_movies.id == df_performance.movie_id,'inner')
df_silver = df_silver.join(df_public_response,df_silver.id == df_public_response.movie_id,'inner')

##Grouping Data by Movies

In [0]:
df_gold = df_silver.groupBy('id','name').agg(
    sum(col('domestic_box_office') + col('international_box_office')).alias('Total_Box_Office'),
    sum('audience_score').alias('Audience'),
    max('cinema_score').alias('Score'),
    avg('metacritical').alias('Critival')
    )

##Writing Delta Format Gold Layer

In [0]:
df_gold.write.format("delta").mode("overwrite").save("/FileStore/tables/Marvel/Gold/Marvel_Movies_Analysis/")


##Showing Analysis

In [0]:
display(df_gold)

id,name,Total_Box_Office,Audience,Score,Critival
1,Iron Man,585171547,91,A,79.0
2,The Incredible Hulk,265573859,70,A-,61.0
3,Iron Man 2,621156389,71,A,57.0
4,Thor,449326618,76,B+,57.0
5,Captain America: The First Avenger,370569776,75,A-,66.0
6,The Avengers,1515100211,91,A+,69.0
7,Iron Man 3,1215392272,78,A,62.0
8,Thor: The Dark World,644602516,75,A-,54.0
9,Captain America: The Winter Soldier,714401889,92,A,70.0
10,Guardians of the Galaxy,770882395,92,A,76.0
